# Generate Glove Embeddings for pairwise keywords


In [1]:

import numpy as np
import pandas as pd
import os
import time
import spacy 
import scipy
from spacy import displacy
import gensim.downloader as api


# Create the output folder if it doesn't exist
output_folder = os.path.expanduser("~/Desktop/UNGDC/output/embedding_static/glove")
os.makedirs(output_folder, exist_ok=True)

start_time = time.time()

In [2]:
import pandas as pd
import numpy as np
import os
from sklearn.metrics.pairwise import cosine_similarity
import gensim.downloader as api

# Load the input data
df = pd.read_csv('~/Downloads/liwc_meta.csv')

# Load GloVe embeddings (300-dimensional, same underlying Stanford release as
# the old glove.6B.300d.txt pipeline, downloaded via gensim instead of a local file)
print("Loading GloVe embeddings...")
glove = api.load("glove-wiki-gigaword-300")


Loading GloVe embeddings...


In [3]:

import spacy

# Lemmatize speeches (per paper: "all words in the speeches were lemmatized
# to their base forms" before generating embeddings), keep only tokens in the
# GloVe vocabulary, then average each speech's word vectors into one document
# embedding. Cached to disk since lemmatizing ~9.5k speeches is slow (~25 min).
embedding_cache_path = os.path.expanduser(
    "~/Desktop/UNGDC/output/embedding_glove/_cache_doc_embeddings_glove300_lemmatized.npy"
)

if os.path.exists(embedding_cache_path):
    print("Loading cached document embeddings...")
    embeddings = np.load(embedding_cache_path)
else:
    nlp = spacy.load("en_core_web_sm", disable=["parser", "ner"])

    speeches = df['text'].tolist()
    print("Lemmatizing and filtering speeches...")
    filtered_speeches = []
    for doc in nlp.pipe(speeches, batch_size=50):
        lemmas = [token.lemma_.lower() for token in doc if token.is_alpha]
        filtered_speeches.append([word for word in lemmas if word in glove])

    print("Computing document embeddings...")
    embeddings = np.array([
        np.mean([glove[word] for word in speech], axis=0) if speech else np.zeros(glove.vector_size)
        for speech in filtered_speeches
    ])

    os.makedirs(os.path.dirname(embedding_cache_path), exist_ok=True)
    np.save(embedding_cache_path, embeddings)


Loading cached document embeddings...


In [4]:

# Keyword pairs anchored on "sovereignty" (per paper's five dependent variables)
keyword_pairs = [
    ("sovereignty", "colonial"),
    ("sovereignty", "territory"),
    ("sovereignty", "westphalia"),
    ("sovereignty", "intervene"),
    ("sovereignty", "interfere"),
]

# Keep every original column (ccode_iso, year, session, and all country/document
# covariates already merged into liwc_meta.csv) and add one distance column per
# keyword pair, so this file can be used directly without a separate merge step.
output_df = df.copy()

# For each speech, the distance to a keyword pair is the average of the
# speech-to-keyword1 distance and the speech-to-keyword2 distance (per
# footnote 10: distance between the speech and the pair of keywords "as a group").
print("Computing per-document distances for each keyword pair...")
for keyword1, keyword2 in keyword_pairs:
    if keyword1 in glove.key_to_index and keyword2 in glove.key_to_index:
        embedding1 = glove[keyword1].reshape(1, -1)
        embedding2 = glove[keyword2].reshape(1, -1)

        similarity1 = cosine_similarity(embedding1, embeddings)[0]
        similarity2 = cosine_similarity(embedding2, embeddings)[0]
        distance = ((1 - similarity1) + (1 - similarity2)) / 2

        output_df[f'distance_{keyword1}_{keyword2}'] = distance
    else:
        print(f"Keywords '{keyword1}' or '{keyword2}' not found in the GloVe vocabulary.")

# Save the updated DataFrame (wide format, all covariates preserved)
output_folder = os.path.expanduser("~/Desktop/UNGDC/output/embedding_glove/")
os.makedirs(output_folder, exist_ok=True)
output_path = os.path.join(output_folder, "liwc_meta_with_keyword_distances_300d_Jul2026.csv")
output_df.to_csv(output_path, index=False)

print(f"Distances for all keyword pairs saved (wide format, all covariates preserved) as '{output_path}'.")


Computing per-document distances for each keyword pair...


Distances for all keyword pairs saved (wide format, all covariates preserved) as '/Users/jihyeonbae/Desktop/UNGDC/output/embedding_glove/liwc_meta_with_keyword_distances_300d_Jul2026.csv'.
